# Baseline Model

SECOM 데이터셋에서 별도의 성능 향상 기법을 적용하지 않은 기준(Baseline) 모델을 구축한다.

이후 수행할 결측치 처리, 클래스 불균형 처리, Feature Selection 등의 효과를 객관적으로 비교하기 위한 기준 성능을 확보하는 것을 목표로 한다.

## 1. Library Import

In [1]:
import pandas as pd
from modeling.models import get_models
from modeling.metrics import calculate_metrics
from preprocessing.pipeline import build_pipeline
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from data.loader import load_dataset
from analysis.profiler import DataProfiler

## 2. Data Load

In [2]:
df = load_dataset("uci-secom.csv")
df.head()

,Time,0,1,2,3,4,5,6,7,8,...,581,582,583,584,585,586,587,588,589,Pass/Fail
0,2008-07-19 11:55:00,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,...,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,-1
1,2008-07-19 12:32:00,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,...,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,-1
2,2008-07-19 13:17:00,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,...,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1
3,2008-07-19 14:43:00,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,...,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,-1
4,2008-07-19 15:22:00,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,...,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,-1


## 3. Constant Feature Remove

In [3]:
profiler = DataProfiler(df)

feature_summary = profiler.features_summary()

constant_features = feature_summary[feature_summary["constant"]].index.tolist()

df_constant_removed = df.drop(
    columns=constant_features
    )

print(f"Before : {df.shape[1]} features")
print(f"After  : {df_constant_removed.shape[1]} features")

Before : 592 features
After  : 476 features


## 4. Data Split

In [4]:
X = df.drop(columns=["Time", "Pass/Fail"])
y = df["Pass/Fail"]

In [5]:
y = y.replace({
    -1: 0,
     1: 1,
    })

### Target Encoding

- 원본 데이터의 타겟은 Pass=-1, Fail=1로 구성되어 있다.
- XGBoost를 비롯한 일부 모델은 0과 1 형태의 라벨을 요구한다.
- 모든 모델에서 동일한 타겟 체계를 사용하기 위해 Pass를 0, Fail을 1로 변환하였다.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
    )

In [7]:
print(f"Train : {X_train.shape}")
print(f"Test  : {X_test.shape}")

print()

print("-----Train-----\n", y_train.value_counts(), y_train.value_counts(normalize=True))

print()

print("-----Test-----\n", y_test.value_counts(), y_test.value_counts(normalize=True))

Train : (1253, 590)
Test  : (314, 590)

-----Train-----
 Pass/Fail
0    1170
1      83
Name: count, dtype: int64 Pass/Fail
0    0.933759
1    0.066241
Name: proportion, dtype: float64

-----Test-----
 Pass/Fail
0    293
1     21
Name: count, dtype: int64 Pass/Fail
0    0.933121
1    0.066879
Name: proportion, dtype: float64


### Interpretation

- 데이터를 Train(80%)과 Test(20%)로 분리하였다.
- Stratified Sampling을 적용하여 Train과 Test 모두 원본 데이터와 동일한 클래스 비율을 유지하였다.
- 불균형 데이터에서는 Stratified Sampling을 적용하지 않으면 Test 데이터에 소수 클래스가 지나치게 적게 포함될 수 있으므로 이를 방지하였다.
- 이후 모든 Feature Selection과 전처리는 Train 데이터로만 학습(fit)하고 Test 데이터에는 변환(transform)을 적용한다.

## 5. Baseline Models

- Constant Feature를 제거한 데이터셋을 사용하였다.
- 결측치는 Median Imputation으로 처리하였다.
- 거리 기반 및 선형 모델의 안정적인 학습을 위해 Logistic Regression에는 StandardScaler를 적용하였다.
- 트리 기반 모델(Random Forest, XGBoost)은 스케일링의 영향을 거의 받지 않으므로 적용하지 않았다.
- 모든 모델은 기본 하이퍼파라미터(Default Parameter)로 학습하여 이후 실험의 기준(Baseline)으로 사용하였다.
- Baseline 모델의 성능은 이후 SMOTE, Variance Threshold, Imputation 방법 변경 등의 효과를 비교하기 위한 기준으로 사용한다.
- 성능평가는 불균형 데이터셋이므로 Accuracy보다 Recall과 F1-score를 중심으로 성능을 평가한다.

### Median & mean

In [8]:
imputers = [
    "mean",
    "median",
    ]

results = []

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for strategy in imputers:

    for model_name, model_info in get_models().items():

        fold_metrics = []
        
        for train_idx, val_idx in cv.split(X_train, y_train):

            pipeline = build_pipeline(
                model=model_info,
                imputer=strategy,
                scaler="standard",
            )
            
            X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
            y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

            pipeline.fit(X_tr, y_tr)
            metrics = calculate_metrics(pipeline, X_val, y_val)

            fold_metrics.append(metrics)

        avg_metrics = pd.DataFrame(fold_metrics).mean(numeric_only=True).to_dict()
        metrics["experiment"] = strategy
        metrics["method"] = strategy
        metrics["model"] = model_name

        results.append(metrics)

missing_value_results = pd.DataFrame(results)

missing_value_results

,accuracy,precision,recall,f1,confusion_matrix,roc_auc,experiment,method,model
0,0.888,0.25000,0.375,0.300000,"[[216, 18], [10, 6]]",0.662660,mean,mean,Logistic Regression
1,0.936,0.00000,0.000,0.000000,"[[234, 0], [16, 0]]",0.710470,mean,mean,Random Forest
2,0.936,0.00000,0.000,0.000000,"[[234, 0], [16, 0]]",0.627137,mean,mean,XGBoost
3,0.892,0.26087,0.375,0.307692,"[[217, 17], [10, 6]]",0.668269,median,median,Logistic Regression
4,0.936,0.00000,0.000,0.000000,"[[234, 0], [16, 0]]",0.669204,median,median,Random Forest
5,0.936,0.00000,0.000,0.000000,"[[234, 0], [16, 0]]",0.633013,median,median,XGBoost


In [9]:
missing_value_results = (
    missing_value_results
    .set_index(
        ["experiment", "method", "model"]
        )
    )

missing_value_results

accuracy  precision  recall        f1  \
experiment method model                                                        
mean       mean   Logistic Regression     0.888    0.25000   0.375  0.300000   
                  Random Forest           0.936    0.00000   0.000  0.000000   
                  XGBoost                 0.936    0.00000   0.000  0.000000   
median     median Logistic Regression     0.892    0.26087   0.375  0.307692   
                  Random Forest           0.936    0.00000   0.000  0.000000   
                  XGBoost                 0.936    0.00000   0.000  0.000000   

                                           confusion_matrix   roc_auc  
experiment method model                                                
mean       mean   Logistic Regression  [[216, 18], [10, 6]]  0.662660  
                  Random Forest         [[234, 0], [16, 0]]  0.710470  
                  XGBoost               [[234, 0], [16, 0]]  0.627137  
median     median Logistic Regression  [[217, 17], [10, 6]]  0.668269  
                  Random Forest         [[234, 0], [16, 0]]  0.669204  
                  XGBoost               [[234, 0], [16, 0]]  0.633013

### Interpretation

- Logistic Regression에서는 Median이 Mean보다 F1 및 ROC-AUC에서 소폭 우수했다.

- 그러나 Random Forest에서는 Mean이 Median보다 ROC-AUC에서 약 0.041p 높은 성능을 보였다.

- XGBoost에서는 두 방식의 차이가 크지 않았다.

- 따라서 전체 모델의 성능을 종합적으로 고려할 경우 Mean imputation이 보다 안정적인 기본 설정으로 판단된다.

- 특히 Random Forest의 ROC-AUC가 상대적으로 크게 개선되었으므로, 모델 간 공통 전처리 설정을 선정한다는 현재 실험 목적에서는 Mean을 우선 후보로 채택한다.

### Decision

- 아래와 같은 Baseline을 기준으로 이후 실험을 진행한다.
- 이후 실험에서는 동일한 Train/Test Split Data를 유지하며 비교한다.

    | 항목                | 설정       |
    | ------------------ | -------    |
    | Constant Feature   | 제거       |
    | Missing Value      | Mean       |
    | Scaling            | 모델별 적용 |
    | SMOTE              | 미적용     |
    | Variance Threshold | 미적용     |
    | Hyperparameter     | Default    |

In [18]:
df_constant_removed.to_csv("../data/processed/uci-secom-constant-removed.csv", index=False)